# Differentially Private LoRA Fine-Tuning of Causal Language Models

Walk through how to fine-tune a GPT-style causal language model on a small generation task with differential privacy, using parameter-efficient LoRA adapters. We compare a non-private LoRA baseline against the DP-LoRA variant and report BLEU, perplexity, peak memory, and throughput for each.

Tracking: [meta-pytorch/opacus#827](https://github.com/meta-pytorch/opacus/issues/827)

## What you'll get out of this notebook

1. A working recipe for combining `opacus`, `peft`, and HuggingFace `transformers` on a causal LM
2. The three device/training-mode ordering patterns that prevent silent corruption (see [#820](https://github.com/meta-pytorch/opacus/issues/820))
3. A concrete sense of the utility cost of DP at a fixed privacy budget on this task
4. Notes on current limitations of full-finetune DP on GPT-2 (out of scope for this tutorial, recipe documented at the end)

## Prerequisites

- Familiarity with the standard DP-SGD setup (`PrivacyEngine`, `make_private_with_epsilon`). The opacus [Building text classifier tutorial](https://github.com/meta-pytorch/opacus/blob/main/tutorials/building_text_classifier.ipynb) is a good warm-up.
- Some prior exposure to LoRA. The [original paper](https://arxiv.org/abs/2106.09685) is short and worth reading.

## Environment and runtime

- Single GPU is sufficient (Kaggle T4 used here)
- Roughly 15 minutes end-to-end on T4 at the settings below (most of it is the 2000-step training pass)
- Pinned versions: `opacus>=1.6.0`, `peft>=0.18,<0.19`, `transformers>=5.0`

## Why combine DP-SGD with LoRA?

DP-SGD adds calibrated Gaussian noise to the per-sample gradient sum at each step. The noise scale grows with the L2 sensitivity of the per-sample gradient (the clipping threshold), so a model with fewer trainable parameters effectively reduces the noise injected into the *learned* parameters. LoRA is a natural fit: it constrains updates to a small low-rank subspace alongside the frozen base weights. In our experiments below, LoRA trains roughly **0.47%** of the GPT-2-small parameters and still reaches a respectable BLEU on the E2E NLG benchmark, both with and without DP.

The combination also lines up cleanly with how practitioners deploy DP today. Sensitive training data warrants a real privacy guarantee; production training budgets warrant parameter-efficient methods. DP-LoRA is the intersection.

## The task: E2E NLG

[E2E NLG](https://arxiv.org/abs/1706.09254) is a small structured-data-to-text generation task originally from the 2017 E2E challenge. Inputs are slot-value meaning representations such as `name[The Vaults], eatType[pub], priceRange[more than £30]`; outputs are short natural-language descriptions. The dataset is small enough to fine-tune in minutes on a single GPU but rich enough that BLEU separates a trained model from chance.

It is also one of the benchmarks used in the [DiSK paper](https://arxiv.org/abs/2410.03883) and adjacent DP-NLP work, which makes results here comparable to the published literature.

## 1. Install pinned versions

`peft>=0.18` requires `transformers>=5.0`. The `opacus>=1.6.0` pin is for the version we developed against; older opacus may also work but the integration patterns below assume 1.6+.

In [1]:
!pip install --quiet \
    'opacus>=1.6.0' \
    'peft>=0.18,<0.19' \
    'transformers>=5.0' \
    'accelerate' \
    'datasets' \
    'evaluate' \
    'sacrebleu'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00


## 2. Sanity-check the environment

If CUDA reports unavailable, enable a GPU accelerator before proceeding. The assertions guard against pip silently resolving to incompatible versions.

In [2]:
import torch
import opacus
import peft
import transformers
import datasets

print(f'torch        = {torch.__version__}  (CUDA: {torch.cuda.is_available()})')
print(f'opacus       = {opacus.__version__}')
print(f'peft         = {peft.__version__}')
print(f'transformers = {transformers.__version__}')
print(f'datasets     = {datasets.__version__}')

if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')


torch        = 2.10.0+cu128  (CUDA: True)
opacus       = 1.6.0
peft         = 0.18.1
transformers = 5.0.0
datasets     = 5.0.0
device       = Tesla T4


## 3. Imports and device

Standard imports plus a seed for partial reproducibility (note: data-loader shuffling and DP noise are still stochastic across runs).

In [3]:
import math
import time
from dataclasses import dataclass, field
from typing import Optional

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from opacus import PrivacyEngine
from datasets import load_dataset
import evaluate

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


Using device: cuda


## 4. Load E2E NLG

The HuggingFace Hub mirrors for E2E NLG (`tuetschek/e2e_nlg`, `GEM/e2e_nlg`) both ship Python loading scripts, which `datasets>=3.0` no longer supports. The dataset is just two CSVs in the upstream repo, so we load them directly with `pandas` and wrap as a `DatasetDict`. Schema after rename: `meaning_representation` (input), `target` (single human reference).

The dev split contains multiple references per MR represented as multiple rows; we treat each row as an independent training example here. A higher-fidelity BLEU evaluation would aggregate references per MR; see the follow-up section at the end.

In [4]:
import pandas as pd
from datasets import Dataset, DatasetDict

E2E_TRAIN_URL = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/trainset.csv'
E2E_DEV_URL   = 'https://raw.githubusercontent.com/tuetschek/e2e-dataset/master/devset.csv'

df_train = pd.read_csv(E2E_TRAIN_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})
df_val   = pd.read_csv(E2E_DEV_URL).rename(columns={'mr': 'meaning_representation', 'ref': 'target'})

ds = DatasetDict({
    'train': Dataset.from_pandas(df_train),
    'validation': Dataset.from_pandas(df_val),
})
print(ds)
print()
print('--- Sample (train) ---')
print('MR:    ', ds['train'][0]['meaning_representation'])
print('Target:', ds['train'][0]['target'])


DatasetDict({
    train: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 42061
    })
    validation: Dataset({
        features: ['meaning_representation', 'target'],
        num_rows: 4672
    })
})

--- Sample (train) ---
MR:     name[The Vaults], eatType[pub], priceRange[more than £30], customer rating[5 out of 5], near[Café Adriatic]
Target: The Vaults pub near Café Adriatic has a 5 star rating.  Prices start at £30.


### Subsample validation for fast eval

Use the full train split (about 42K rows). Subsample the validation split to 200 rows so the BLEU sweep across configurations stays under a couple of minutes.

In [5]:
# Use the full E2E NLG train split. For evaluation, group the dev split by
# meaning representation: E2E provides several human references per MR, and the
# official metric protocol scores generations against all of them.
EVAL_MRS = 100

ds_train_small = ds['train']
ds_val_small = ds['validation'].shuffle(seed=42).select(range(200))  # for perplexity

refs_by_mr = df_val.groupby('meaning_representation')['target'].apply(list)
eval_mrs = refs_by_mr.sample(n=EVAL_MRS, random_state=42)
print(f'Train examples: {len(ds_train_small)}')
print(f'Eval MRs: {len(eval_mrs)} (refs per MR: min={eval_mrs.str.len().min()}, '
      f'max={eval_mrs.str.len().max()})')


Train examples: 42061
Eval MRs: 100 (refs per MR: min=4, max=37)


## 5. Tokenize for causal-LM training

Format each row as `'{MR} -> {target}'` and tokenize with GPT-2's tokenizer to a fixed length. We set `labels = input_ids` (predict every position, including the prompt portion). This trains the model slightly differently from a more typical "loss only on the target" setup, but it sidesteps an interaction between `-100`-masked labels, padding, and opacus's per-sample-gradient tracking. See the safety-patterns section below for the full rationale.

In [6]:
MODEL_NAME = 'gpt2'
MAX_SEQ_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

PROMPT_TEMPLATE = '{mr} ->'

def tokenize_example(example):
    """Format as 'MR -> target', tokenize, pad to fixed length.

    labels = input_ids, so the model trains on the whole sequence rather than
    only the target span. Masking the prompt with -100 is more sample-efficient,
    but combined with padding and LoRA it triggers a per-sample-gradient shape
    mismatch in opacus, so we keep the simpler formulation here. A custom
    collator that masks the prompt without tripping that path would be a
    reasonable improvement.
    """
    prompt = PROMPT_TEMPLATE.format(mr=example['meaning_representation'])
    target = ' ' + example['target'] + tokenizer.eos_token
    full_text = prompt + target

    enc = tokenizer(
        full_text,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
        truncation=True,
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': enc['input_ids'],
    }

ds_train_tok = ds_train_small.map(tokenize_example, remove_columns=ds_train_small.column_names)
ds_val_tok = ds_val_small.map(tokenize_example, remove_columns=ds_val_small.column_names)
ds_train_tok.set_format('torch')
ds_val_tok.set_format('torch')
print(f'Tokenized train: {ds_train_tok}')
print(f'Tokenized val:   {ds_val_tok}')


07/20/2026 11:48:48:WARNING:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/42061 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenized train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 42061
})
Tokenized val:   Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 200
})


## 6. The three safety patterns

These are the device-placement and training-state orderings that prevent silent corruption when combining HuggingFace + PEFT + opacus on a causal LM. Each of them addresses a real failure mode that we ran into while building this tutorial; we document them inline so readers do not have to rediscover them.

### Pattern A: `model.to(device)` before `get_peft_model()`

With newer PEFT (`>=0.18`), accelerate-style lazy device handling can leave parts of the model on CPU when opacus walks `add_hooks()`. The symptoms are subtle: training loss looks reasonable, the privacy accountant ticks, but LoRA weights never update. We confirmed this empirically across three independent setups (CPU bisect across peft 0.13.2 → 0.18.1, Kaggle T4, RTX 5090) in [opacus#820](https://github.com/meta-pytorch/opacus/issues/820). The safe order is:

```python
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)                # ← move base model to CUDA FIRST
model = get_peft_model(model, config)   # then apply LoRA (LoRA params land on CUDA too)
# then PrivacyEngine.make_private(...)
```

### Pattern B: `model.train()` before `make_private_with_epsilon()`

opacus's `ModuleValidator` (1.6.0+) raises `IllegalModuleConfigurationError("Model needs to be in training mode")` if the model is not in train mode when `make_private_with_epsilon` runs. `get_peft_model()` puts the model in eval mode by default, so an explicit `model.train()` call between PEFT wrapping and opacus wrapping is required.

### Pattern C: `poisson_sampling=False`

opacus's default Poisson sampling occasionally produces an empty batch (probability around `e^(-batch_size)` per step). GPT-2's forward pass calls `attention_mask.view(batch_size, -1)`, which fails for a zero-element tensor because the `-1` dimension is ambiguous. Setting `poisson_sampling=False` switches to uniform-without-replacement sampling, which has deterministic batch sizes and avoids the edge case. Uniform sampling with DP-SGD is commonly used in practice.

One caveat to be aware of: the accountant's privacy analysis assumes Poisson sampling. opacus emits a warning when Poisson sampling is disabled, and the reported ε under uniform sampling should be treated as an approximation rather than an exact guarantee (Chua et al., ICML 2024, [arXiv:2403.17673](https://arxiv.org/abs/2403.17673) quantify the gap between the two sampling schemes). Keep the warning visible in your own runs instead of suppressing it.


## 7. Run configuration

Encapsulate the per-run hyperparameters in a small dataclass so the same training loop can drive all configurations.

In [7]:
@dataclass
class RunConfig:
    name: str
    lora: bool
    dp: bool
    lr: float = 1e-4
    batch_size: int = 8
    epochs: int = 2
    target_epsilon: float = 8.0
    target_delta: float = 1e-5
    max_grad_norm: float = 1.0
    lora_r: int = 16
    lora_alpha: int = 32


## 8. Build the model for a given configuration

Single entry point that handles base-model loading, optional LoRA wrapping, and the Pattern A ordering. The `cfg.dp and not cfg.lora` branch applies `ModuleValidator.fix()` to swap GPT-2's `transformers.Conv1D` modules for `nn.Linear`; this is required when the DP-full path is enabled (which we do not enable in this tutorial — see the follow-up section).

In [8]:
def build_model_for_config(cfg: RunConfig):
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    # DP-full path: swap GPT-2's transformers.Conv1D modules with nn.Linear so opacus's
    # per-sample-gradient hooks attach correctly. opacus has a registered fix for Conv1D.
    # No-op for the LoRA path (LoRA wraps Conv1D with its own A/B Linear modules, which
    # opacus already handles).
    if cfg.dp and not cfg.lora:
        from opacus.validators import ModuleValidator
        model = ModuleValidator.fix(model)
        print(f'[{cfg.name}] Applied ModuleValidator.fix() (Conv1D -> nn.Linear)')

    model = model.to(device)  # <-- BEFORE PEFT, per #820

    if cfg.lora:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=cfg.lora_r,
            lora_alpha=cfg.lora_alpha,
            lora_dropout=0.0,
            target_modules=['c_attn'],
        )
        model = get_peft_model(model, lora_config)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'[{cfg.name}] trainable={trainable:,} ({100*trainable/total:.2f}% of {total:,})')
    return model


## 9. The shared training function

Same loop for all configurations. The `cfg.dp` block adds the `PrivacyEngine`, applies the three safety patterns, and uses `poisson_sampling=False` per Pattern C. Memory and throughput are tracked across the run; the final epsilon is read from the accountant at the end. The per-step skip on empty batches is a defensive guard that should never actually fire with `poisson_sampling=False`.

In [9]:
def train_one_run(cfg: RunConfig, train_ds):
    print(f'\n=== Training: {cfg.name} ===')
    model = build_model_for_config(cfg)
    model.train()  # opacus's validator requires train mode before make_private_with_epsilon

    optimizer = optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=cfg.lr,
    )
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)

    privacy_engine = None
    if cfg.dp:
        privacy_engine = PrivacyEngine(accountant='rdp')
        # poisson_sampling=False: uniform-without-replacement batches; see Pattern C
        # above for why, including the caveat on the accountant's analysis.
        model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader,
            target_epsilon=cfg.target_epsilon,
            target_delta=cfg.target_delta,
            epochs=cfg.epochs,
            max_grad_norm=cfg.max_grad_norm,
            poisson_sampling=False,
        )
        print(f'[{cfg.name}] noise_multiplier={optimizer.noise_multiplier:.4f}')

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t_start = time.perf_counter()
    tokens_processed = 0

    model.train()
    step = 0
    loss_history = []
    for epoch in range(cfg.epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_history.append(loss.item())
            tokens_processed += batch['input_ids'].numel()
            if step % 500 == 0:
                print(f'  epoch {epoch} step {step:5d}  loss={loss.item():.4f}')
            step += 1

    t_total = time.perf_counter() - t_start
    peak_mem_gb = (torch.cuda.max_memory_allocated() / 1024**3) if torch.cuda.is_available() else 0.0
    final_epsilon = privacy_engine.get_epsilon(delta=cfg.target_delta) if cfg.dp else float('inf')

    return {
        'config': cfg,
        'model': model,
        'mean_loss': sum(loss_history) / len(loss_history),
        'final_loss': loss_history[-1],
        'tokens_per_sec': tokens_processed / t_total,
        'peak_mem_gb': peak_mem_gb,
        'wall_clock_sec': t_total,
        'epsilon': final_epsilon,
        'steps_completed': step,
    }


## 10. Evaluation: multi-reference BLEU plus perplexity

E2E provides several human references per meaning representation, and published results (DiSK, Yu et al. 2022) follow the official protocol of scoring against all of them. We generate with beam search (`num_beams=5`) from each held-out MR and compute corpus-level sacreBLEU against the full reference set. Perplexity is computed separately on the tokenized validation split. Single-reference greedy scoring understates BLEU on this task by a wide margin and is not comparable to published numbers, so it is worth matching the official protocol when reporting results.

In [10]:
bleu_metric = evaluate.load('sacrebleu')

def evaluate_run(model, eval_mrs_series, val_ds_tok, max_new_tokens=64):
    """Multi-reference corpus BLEU with beam decoding, plus perplexity.

    eval_mrs_series: pandas Series indexed by meaning_representation, values are
    lists of human references (from the grouped dev split).
    """
    gen_model = model._module if hasattr(model, '_module') else model
    gen_model.eval()

    preds, refs = [], []
    max_refs = eval_mrs_series.str.len().max()

    with torch.no_grad():
        for mr, references in eval_mrs_series.items():
            prompt = PROMPT_TEMPLATE.format(mr=mr)
            prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            gen = gen_model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                num_beams=5,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
            pred = tokenizer.decode(gen[0, prompt_ids.shape[1]:], skip_special_tokens=True).strip()
            preds.append(pred)
            # sacrebleu requires the same reference count per prediction; pad by
            # repeating the first reference (standard practice, does not change BLEU)
            padded = references + [references[0]] * (max_refs - len(references))
            refs.append(padded)

        # perplexity on the tokenized validation split
        val_losses = []
        for i in range(min(200, len(val_ds_tok))):
            ex = val_ds_tok[i]
            out = gen_model(
                input_ids=ex['input_ids'].unsqueeze(0).to(device),
                attention_mask=ex['attention_mask'].unsqueeze(0).to(device),
                labels=ex['labels'].unsqueeze(0).to(device),
            )
            val_losses.append(out.loss.item())

    bleu = bleu_metric.compute(predictions=preds, references=refs)['score']
    perplexity = math.exp(sum(val_losses) / len(val_losses))
    return {'bleu': bleu, 'perplexity': perplexity, 'n_eval': len(preds)}


## 11. Configurations to compare

Four configurations. The non-DP LoRA baseline and one DP-LoRA setting at a higher learning rate, plus two DP-LoRA ablations that keep the non-private learning rate and instead adjust batch size or the clipping threshold.

A note on the learning rate, since it deserves care. With `max_grad_norm=1.0`, per-sample gradients early in training are heavily clipped, which attenuates the average update well below its non-private magnitude. Something has to compensate: a larger learning rate, a larger batch size (which improves the signal-to-noise ratio of the noisy mean), or a higher clipping threshold (which attenuates less). The DP fine-tuning literature reports learning rates roughly an order of magnitude above typical non-private values for this kind of setup; see [Li et al., ICLR 2022](https://arxiv.org/abs/2110.05679) on the interplay of learning rate and batch size for DP LM fine-tuning, and [Yu et al., ICLR 2022](https://arxiv.org/abs/2110.06500) for DP parameter-efficient fine-tuning of GPT-2 on E2E specifically. The ablations below let you compare the three compensation strategies directly.

In [11]:
# batch_size=64 for the larger-batch ablation exceeds 16 GB on a T4; 32 is the
# largest power-of-two that fits alongside opacus's per-sample gradients here.
CONFIGS = [
    RunConfig(name='non-DP LoRA',      lora=True, dp=False, lr=1e-4, batch_size=8,  epochs=2),
    RunConfig(name='DP LoRA lr=5e-4',  lora=True, dp=True,  lr=5e-4, batch_size=8,  epochs=2),
    RunConfig(name='DP LoRA bs=32',    lora=True, dp=True,  lr=1e-4, batch_size=32, epochs=2),
    RunConfig(name='DP LoRA clip=3.0', lora=True, dp=True,  lr=1e-4, batch_size=8,  epochs=2, max_grad_norm=3.0),
]


## 12. Run the configurations

Roughly 40 minutes per configuration on a T4 at these settings (two epochs over the full train split), so expect the full sweep to take a few hours. Memory is freed between runs.

In [12]:
results = []
for cfg in CONFIGS:
    # Each configuration is isolated: if one runs out of memory on your GPU
    # (the larger-batch setting is the usual culprit), the sweep keeps going
    # and the comparison table reports whatever completed.
    try:
        train_result = train_one_run(cfg, ds_train_tok)
        eval_result = evaluate_run(train_result['model'], eval_mrs, ds_val_tok)
        results.append({**train_result, **eval_result})
        print(f'[{cfg.name}] BLEU={eval_result["bleu"]:.2f}  PPL={eval_result["perplexity"]:.2f}')
        del train_result['model']
    except torch.cuda.OutOfMemoryError:
        print(f'[{cfg.name}] SKIPPED: out of memory. Reduce batch_size for this configuration.')
    finally:
        torch.cuda.empty_cache() if torch.cuda.is_available() else None



=== Training: non-DP LoRA ===


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/opacus/accountants/analysis/rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy b

[DP LoRA clip=3.0] trainable=589,824 (0.47% of 125,029,632)
[DP LoRA clip=3.0] noise_multiplier=0.3828


sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


  epoch 0 step     0  loss=8.7529
  epoch 0 step   500  loss=6.8720
  epoch 0 step  1000  loss=2.8173
  epoch 0 step  1500  loss=2.4631
  epoch 0 step  2000  loss=2.0345
  epoch 0 step  2500  loss=1.9668
  epoch 0 step  3000  loss=1.7959
  epoch 0 step  3500  loss=1.7770
  epoch 0 step  4000  loss=1.9407
  epoch 0 step  4500  loss=1.6403
  epoch 0 step  5000  loss=1.8761
  epoch 1 step  5500  loss=1.5938
  epoch 1 step  6000  loss=1.5857
  epoch 1 step  6500  loss=1.5342
  epoch 1 step  7000  loss=1.2777
  epoch 1 step  7500  loss=1.3622
  epoch 1 step  8000  loss=1.2682
  epoch 1 step  8500  loss=1.2796
  epoch 1 step  9000  loss=1.0052
  epoch 1 step  9500  loss=1.1905
  epoch 1 step 10000  loss=1.2610
  epoch 1 step 10500  loss=0.9629
[DP LoRA clip=3.0] BLEU=38.90  PPL=3.08


## 13. Comparison table

In [13]:
def fmt_eps(e):
    return 'no DP' if e == float('inf') else f'{e:.2f}'

print(f'{"Config":<14} | {"BLEU":>6} | {"PPL":>7} | {"tok/s":>8} | {"peak GB":>7} | {"sec":>6} | {"ε":>6}')
print('-' * 75)
for r in results:
    print(f'{r["config"].name:<14} | {r["bleu"]:>6.2f} | {r["perplexity"]:>7.2f} | '
          f'{r["tokens_per_sec"]:>8.0f} | {r["peak_mem_gb"]:>7.2f} | '
          f'{r["wall_clock_sec"]:>6.1f} | {fmt_eps(r["epsilon"]):>6}')


Config         |   BLEU |     PPL |    tok/s | peak GB |    sec |      ε
---------------------------------------------------------------------------
non-DP LoRA    |  68.38 |    1.50 |     4776 |    2.17 | 2254.3 |  no DP
DP LoRA lr=5e-4 |  47.13 |    2.02 |     4704 |    2.65 | 2288.8 |   8.00
DP LoRA bs=32  |  44.88 |    2.81 |     4652 |    8.16 | 2314.5 |   7.99
DP LoRA clip=3.0 |  38.90 |    3.08 |     4705 |    3.66 | 2288.3 |   8.00


## 14. Reading the results

Sample run on a Kaggle T4: GPT-2-small with LoRA on `c_attn`, two epochs over the full E2E NLG train split, multi-reference BLEU with beam-5 decoding over 100 held-out meaning representations.

| Config | BLEU | PPL | tok/s | Peak GB | Time | ε |
|---|---|---|---|---|---|---|
| non-DP LoRA (lr 1e-4) | 68.38 | 1.50 | 4776 | 2.17 | 2254 s | no DP |
| DP LoRA (lr 5e-4) | 47.13 | 2.02 | 4704 | 2.65 | 2289 s | 8.00 |
| DP LoRA (lr 1e-4, batch 32) | 44.88 | 2.81 | 4652 | 8.16 | 2315 s | 7.99 |
| DP LoRA (lr 1e-4, clip 3.0) | 38.90 | 3.08 | 4705 | 3.66 | 2288 s | 8.00 |

LoRA trains 589,824 parameters, about 0.47% of GPT-2-small's 125M, and the non-private run reaches BLEU 68.38. That is the parameter-efficiency argument in a single number.

Differential privacy at ε = 8 costs roughly 31% relative BLEU here, 68.38 down to 47.13 for the best DP configuration, with perplexity moving from 1.50 to 2.02. The model still learns the conditional generation task; the noise and clipping shift utility down without collapsing it.

### Three ways to pay for clipping, and they are not equal

The three DP rows differ only in which hyper-parameter compensates for gradient clipping. With `max_grad_norm=1.0`, per-sample gradients are clipped hard enough early in training that the averaged update lands well below its non-private magnitude, and something has to make up the difference:

- raising the learning rate fivefold, to 5e-4, recovers the most utility (47.13)
- raising the batch size fourfold, to 32, recovers nearly as much (44.88) while holding the learning rate at the non-private 1e-4
- raising the clipping threshold to 3.0 helps least (38.90)

The first two are close enough that either is a reasonable starting point, and a sweep over both together would likely beat either alone. A larger batch improves the signal-to-noise ratio of the noisy gradient mean, which is a different mechanism from simply taking bigger steps, so the two are complementary rather than redundant.

It is worth seeing how badly the uncompensated setting fails. At `lr=1e-4` with `max_grad_norm=1.0`, the same configuration produced BLEU near zero while still reaching a plausible-looking perplexity. Perplexity alone will not tell you the conditional task has failed to train.

### Memory is where the batch-size route gets expensive

Peak memory for the batch-32 run is 8.16 GB against 2.65 GB at batch 8, and a batch of 64 exceeded the 16 GB available on a T4 once opacus's per-sample gradients were accounted for. If you take the larger-batch route to compensate for clipping, budget for it.

One caveat on those memory figures: they are recorded per configuration inside a single process, and the two batch-8 DP runs report 2.65 GB and 3.66 GB despite being identical in batch size. The clipping threshold does not affect allocation, so the difference reflects allocator state carried over from the preceding run rather than a real cost. Treat the numbers as indicative, and measure in isolation if you need them to be precise.

### On comparing against published results

Published E2E numbers generally come from larger GPT-2 variants, full fine-tuning rather than LoRA, and longer schedules, so the figures above are not directly comparable to them. The useful signal is that the non-private 68.38 sits in the range typically reported for well-trained models on this benchmark, which says the evaluation protocol here is sound. An earlier version of these experiments scored the same trained model at 24.72 using single-reference BLEU with greedy decoding; the entire difference came from the scoring protocol, not from the model.


## 15. When to reach for DP-LoRA

A practical heuristic, given the numbers above:

Use DP-LoRA when the training data is sensitive enough to warrant a real privacy guarantee and the downstream task tolerates a meaningful utility drop relative to a non-DP baseline. The utility cost of DP at this budget is quantified in the results table above; for published reference points on this task, see Yu et al. (ICLR 2022).

Consider full DP fine-tuning instead of DP-LoRA when LoRA's restricted parameter subspace is the bottleneck (not the privacy budget) and you have compute headroom to handle opacus's per-sample-gradient memory overhead at the full model scale. The full-finetune path on GPT-2 specifically also needs the engineering steps in the follow-up section below.

## 16. Follow-up work

### Enabling DP-full fine-tuning on GPT-2

Full fine-tuning of GPT-2 under opacus fails out of the box with a shape mismatch in `clip_and_accumulate` (`stack expects each tensor to be equal size`). While preparing this tutorial we initially attributed the failure to GPT-2's tied embedding and output projection. That turned out to be wrong: untying the weights changes nothing.

The actual cause is narrower and easy to check. Transformers builds `position_ids` with shape `[1, seq_len]` and broadcasts them across the batch. Opacus sizes each per-sample gradient from the batch dimension of the activations reaching that module, so the position embedding `transformer.wpe.weight` receives a `grad_sample` with batch dimension 1 while every other parameter receives the true batch size, and the two cannot be stacked.

The workaround is one line in the training loop: pass explicit per-sample position ids.

```python
B, T = input_ids.shape
position_ids = torch.arange(T).unsqueeze(0).expand(B, T)
outputs = model(input_ids=input_ids, position_ids=position_ids, labels=input_ids)
```

With that change, DP full fine-tuning of GPT-2 trains normally. The same pattern should apply to other models with learned absolute position embeddings that broadcast their position ids. A library-level fix in opacus's embedding grad sampler is under discussion.

### Other polishes worth doing

- Full validation set (about 4K MRs) for higher-confidence BLEU; this notebook scores a 100-MR sample to keep runtime reasonable
- 2 to 3 seeds per configuration to quantify variance
- Small HP sweep around `lr`, `max_grad_norm`, and `target_epsilon` to pick "reasonable" settings rigorously
- Loss-only-on-target labels (with `-100` masking on the prompt portion) once the opacus + LoRA per-sample-gradient interaction with masked labels is resolved

### References

- [opacus#820](https://github.com/meta-pytorch/opacus/issues/820) — the device-placement-ordering issue that motivates Pattern A
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- [Deep Learning with Differential Privacy (Abadi et al. 2016)](https://arxiv.org/abs/1607.00133)
- [E2E NLG Challenge dataset](https://arxiv.org/abs/1706.09254) and the upstream [GitHub repo](https://github.com/tuetschek/e2e-dataset)
- [DiSK: Differentially Private Optimizer with Simplified Kalman Filter (Zhang et al. 2024)](https://arxiv.org/abs/2410.03883), which uses E2E NLG as part of its benchmark suite